# Paper experiments on Colab

Runs the full corrected experiment plan (`configs/paper.yaml`) with `scripts/run_paper.py`.

* **Resumable:** checkpoints, the split file and step markers live on Google Drive. After a disconnect,
  re-run the setup cells and then the same stage cell; finished steps are skipped and training continues
  from `last.pt`.
* **GPU:** a T4 is enough. The thesis ran the same model sizes (SwinUNETR, feature size 48, 96³ crops,
  batch 2) on Colab's 15 GB T4, and mixed precision (fp16) plus gradient checkpointing are on.
  A100/L4 is roughly 2–3× faster. The whole plan needs about 70–90 T4-hours; see `docs/paper_protocol.md`.
* **Free-tier Colab** sessions end after at most ~12 h (often sooner, and when idle), and GPU time is
  rationed, so expect many sessions. That is fine: re-run setup + the stage cell and it resumes.
* Stages run in order: splits → SSL → segmentation → predicted masks → features → classification →
  k-fold XGBoost → tables.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Code
Clone your GitHub repo (or copy the folder from Drive) and install it.

In [ ]:
REPO_URL = "https://github.com/kkoutsikos/glioma-mtl.git"
!git clone -q {REPO_URL} /content/glioma-mtl 2>/dev/null || git -C /content/glioma-mtl pull -q
%cd /content/glioma-mtl
!pip install -q -e ".[boosting]"
# PyPI pyradiomics does not build on Colab's Python 3.12; install it from GitHub (as in the thesis notebook).
!pip install -q SimpleITK git+https://github.com/AIM-Harvard/pyradiomics.git
# An editable install is only seen by this notebook after a runtime restart; make it importable now.
import sys
sys.path.insert(0, "/content/glioma-mtl/src")

## 2. Paths
Defaults match your Drive (`Διπλωματικη/Brain Tumor Dataset/`: `volumes`, `seg`,
`UCSF-PDGM-metadata_v5.csv`, `balanced_metadata_gradeaware.csv`). Use the full `volumes` / `seg`
folders (all 495 patients), not `Brain_Tumor_Subset` (162 only). The preflight checks all of this.

In [ ]:
DRIVE = "/content/drive/MyDrive/Διπλωματικη"
IMAGES = f"{DRIVE}/Brain Tumor Dataset/volumes"                          # 495 x *_T2.nii.gz
LABELS = f"{DRIVE}/Brain Tumor Dataset/seg"                              # 495 x *_tumor_segmentation.nii.gz
META = f"{DRIVE}/Brain Tumor Dataset/UCSF-PDGM-metadata_v5.csv"          # TCIA metadata (501 rows)
COHORT = f"{DRIVE}/Brain Tumor Dataset/balanced_metadata_gradeaware.csv" # 162-patient cohort (ID column)
OUT = f"{DRIVE}/paper_runs"                                              # results, best.pt checkpoints, markers
COPY_TO_LOCAL_DISK = True                                                # much faster than reading NIfTI from Drive

### SegMamba (CUDA build, cached on Drive)
SegMamba is not a pip package: its bundled `causal-conv1d` and `mamba` CUDA extensions must be compiled.
The first session builds them (`mamba` takes ~20–40 min) and stores the wheels on Drive. Later sessions
only install the cached wheels (seconds). A new torch/CUDA/Python version on Colab triggers a fresh build
automatically. On failure the cell prints the actual compiler errors (full log in the wheels folder).

**Run it in every session before `predict`, `features`, `cls` or `explain`:** SegMamba is the shared backbone.

In [ ]:
import glob, os, re, subprocess, sys, torch
SEGMAMBA = "/content/glioma-mtl/third_party/SegMamba"
tag = f"torch{torch.__version__.split('+')[0]}_cu{torch.version.cuda}_py{sys.version_info.major}{sys.version_info.minor}"
WHEELS = f"{DRIVE}/wheels/segmamba_{tag}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> switch to a GPU runtime first")
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| Python", sys.version.split()[0])
if not os.path.exists(SEGMAMBA):
    !git clone -q https://github.com/ge-xing/SegMamba.git {SEGMAMBA}
# SegMamba's bundled mamba_ssm imports classes removed from newer transformers (thesis fix). pip then warns
# that gradio / diffusers / sentence-transformers conflict: harmless, the pipeline does not use them.
!pip install -q "transformers==4.39.3"

# SegMamba's 2024 setup.py files hard-code compute_70 (Volta), which CUDA >= 13 can no longer compile.
# Use compute_75 instead: a T4 (7.5) runs it natively, L4 / A100 use the sm_80 / sm_90 targets already there.
for setup in [f"{SEGMAMBA}/causal-conv1d/setup.py", f"{SEGMAMBA}/mamba/setup.py"]:
    src = open(setup).read()
    if "compute_70,code=sm_70" in src:
        open(setup, "w").write(src.replace("compute_70,code=sm_70", "compute_75,code=sm_75"))
        print("patched", setup.split("SegMamba/")[1])

# CUDA 13 ships CCCL 3, which removed cub::CTA_SYNC and cub::LaneId used by mamba's 2024 kernels.
# Replace them with exactly what they were: __syncthreads() and a read of the PTX %laneid register.
LANE_ID = ('#ifndef SEGMAMBA_LANE_ID\n#define SEGMAMBA_LANE_ID\n'
           'static __device__ __forceinline__ unsigned int segmamba_lane_id() '
           '{ unsigned int r; asm volatile("mov.u32 %0, %%laneid;" : "=r"(r)); return r; }\n#endif\n')
for path in glob.glob(f"{SEGMAMBA}/mamba/csrc/**/*.*", recursive=True):
    if not path.endswith((".cu", ".cuh", ".h")):
        continue
    src = open(path).read()
    new = src.replace("cub::CTA_SYNC()", "__syncthreads()")
    if "cub::LaneId()" in new and path.endswith((".cu", ".cuh")):
        new = LANE_ID + new.replace("cub::LaneId()", "segmamba_lane_id()")
    if new != src:
        open(path, "w").write(new)
        print("patched", path.split("SegMamba/")[1])

os.makedirs(WHEELS, exist_ok=True)
def built(prefix):
    return glob.glob(f"{WHEELS}/{prefix}-*.whl")
for pkg, prefix, flag in [("causal-conv1d", "causal_conv1d", "CAUSAL_CONV1D_FORCE_BUILD"),
                          ("mamba", "mamba_ssm", "MAMBA_FORCE_BUILD")]:
    if built(prefix):
        print(f"{pkg}: using cached wheel")
        continue
    log_path = f"{WHEELS}/build_{pkg}.log"
    print(f"{pkg}: building (first time only; mamba takes ~20-40 min). Log: {log_path}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja", "packaging"])
    with open(log_path, "w") as log:
        r = subprocess.run([sys.executable, "-m", "pip", "wheel", "-v", "--no-deps", "--no-build-isolation",
                            "-w", WHEELS, "."], cwd=f"{SEGMAMBA}/{pkg}", stdout=log, stderr=subprocess.STDOUT,
                           env={**os.environ, flag: "TRUE", "MAX_JOBS": "4"})
    if r.returncode != 0:
        lines = open(log_path, errors="replace").read().splitlines()
        distinct = {}  # one example line per distinct compiler error message
        for l in lines:
            m = re.search(r"(error|fatal error): (.*)", l)
            if m and m.group(2) not in distinct:
                distinct[m.group(2)] = l.strip()
        print(f"{pkg} BUILD FAILED. {len(distinct)} distinct compiler errors:\n" + "\n".join(list(distinct.values())[:30])
              + "\n--- last lines of the log ---\n" + "\n".join(lines[-10:]))
        break

wheels = built("causal_conv1d") + built("mamba_ssm")
if len(wheels) < 2:
    raise SystemExit("SegMamba not available yet: paste the errors above. Only the non-SegMamba segmentation runs work without it.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)

# Real check, in a fresh process (like the training run): build through the repo's factory, forward on GPU.
check = '''
import sys, torch
sys.path.insert(0, "/content/glioma-mtl/src")
from glioma_mtl.models import build_segmentation_model
m = build_segmentation_model({"name": "segmamba", "segmamba_repo": "/content/glioma-mtl/third_party/SegMamba"}).cuda().eval()
with torch.no_grad(), torch.autocast("cuda"):
    y = m(torch.randn(1, 1, 96, 96, 96, device="cuda"))
print("SegMamba OK, output shape", tuple(y.shape), f"{sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
'''
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True, cwd="/content/glioma-mtl")
print(r.stdout if r.returncode == 0 else "SegMamba FAILED:\n" + r.stderr[-3000:])

### Preflight (about 1 min): fails here instead of hours into a run

In [ ]:
import glob, os, shutil, subprocess, torch
problems = []
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("GPU:", gpu or "NONE")
if gpu is None:
    problems.append("No GPU: Runtime > Change runtime type > T4 GPU")
for name, path in {"IMAGES": IMAGES, "LABELS": LABELS, "META": META, "COHORT": COHORT}.items():
    if not os.path.exists(path):
        problems.append(f"{name} not found: {path}")
n_img = len(glob.glob(f"{IMAGES}/**/*_T2.nii.gz", recursive=True))
n_seg = len(glob.glob(f"{LABELS}/**/*_tumor_segmentation.nii.gz", recursive=True))
print(f"T2 volumes: {n_img}   masks: {n_seg}   (expected 495 each)")
if n_img < 400 or n_seg < 400:
    problems.append("Fewer volumes/masks than expected; is this the full folder and not Brain_Tumor_Subset?")
import importlib, json, sys
import pandas as pd
if "/content/glioma-mtl/src" not in sys.path:
    sys.path.insert(0, "/content/glioma-mtl/src")
for mod in ["glioma_mtl", "monai", "nibabel", "sklearn", "xgboost", "SimpleITK", "radiomics"]:
    try:
        print(f"{mod:10s} {getattr(importlib.import_module(mod), '__version__', 'ok')}")
    except Exception as e:
        problems.append(f"{mod} not importable ({e}); re-run the install cell and read its errors")
from glioma_mtl.data.metadata import normalize_patient_id
if os.path.exists(META):
    meta = pd.read_csv(META)
    print(f"Metadata: {len(meta)} rows (expected 501 = 495 patients + 6 follow-ups)")
    needed = {"ID", "Age at MRI", "Sex", "WHO CNS Grade", "Final pathologic diagnosis (WHO 2021)", "IDH", "1p/19q"}
    if len(meta) != 501 or not needed <= set(meta.columns):
        problems.append(f"META is not the full UCSF-PDGM v5 metadata (missing columns: {sorted(needed - set(meta.columns))})")
if os.path.exists(COHORT):
    cohort_ids = sorted(pd.read_csv(COHORT)["ID"].map(normalize_patient_id))
    committed = sorted(set().union(*json.load(open("data/splits/splits_seed42.json"))["classification"].values()))
    same = cohort_ids == committed
    print(f"Cohort: {len(cohort_ids)} patients, identical to the committed paper split: {same}")
    if not same:
        problems.append("COHORT differs from the 162-patient cohort in data/splits/splits_seed42.json; "
                        "upload Dipl/splits/balanced_metadata_gradeaware.csv from your laptop")
if not glob.glob("/content/glioma-mtl/third_party/SegMamba/model_segmamba"):
    problems.append("SegMamba not set up: run the SegMamba cell above (the shared backbone: predict, features, cls and explain need it)")
free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive free space: {free_gb:.1f} GB (the plan needs ~4 GB, plus ~1 GB per run in progress)")
if free_gb < 5:
    problems.append("Less than 5 GB free on Drive; delete old thesis runs/ and cache_* folders first")
# The models re-implement SwinUNETR's forward pass; check it matches the installed MONAI version.
r = subprocess.run(["python", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_models.py",
                    "-k", "swin_forward_matches_monai"], capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": "/content/glioma-mtl/src"})
print("MONAI compatibility test:", "passed" if r.returncode == 0 else "FAILED\n" + r.stdout[-2000:])
if r.returncode != 0:
    problems.append("SwinUNETR forward check failed; pin MONAI: pip install 'monai==1.5.*'")
print("\n" + ("ALL GOOD" if not problems else "FIX BEFORE RUNNING:\n- " + "\n- ".join(problems)))

In [ ]:
import os, shlex
if COPY_TO_LOCAL_DISK:
    !mkdir -p /content/data/images /content/data/labels
    !rsync -a --info=stats1 "{IMAGES}/" /content/data/images/
    !rsync -a --info=stats1 "{LABELS}/" /content/data/labels/
    image_dir, label_dir = "/content/data/images", "/content/data/labels"
else:
    image_dir, label_dir = IMAGES, LABELS

overrides = [
    f"data.image_dir={image_dir}", f"data.label_dir={label_dir}",
    f"data.metadata_csv={META}", f"data.cohort_csv={COHORT}",
    f"data.splits_file={OUT}/splits_seed42.json",     # on Drive: the split must never change between sessions
    f"output_root={OUT}",
    # Data loaders scale with the VM: 2 on a free T4, up to 8 on L4/A100 VMs (more CPUs), so a fast GPU isn't starved.
    "data.cache_dir=/content/cache", f"data.num_workers={max(2, min(8, os.cpu_count() or 2))}",
]
SET = " ".join(shlex.quote(o) for o in overrides)
def run(stages, extra="", more=()):
    # `more` overrides come after SET, so they win (used by the pilot below).
    MORE = " ".join(shlex.quote(o) for o in more)
    !python scripts/run_paper.py --plan configs/paper.yaml --stages {stages} {extra} --set {SET} {MORE}

## 3. Check the plan (nothing is executed)

In [ ]:
run("splits ssl seg predict features cls kfold aggregate", "--dry-run")

### Optional pilot (~25 min on a T4)
One epoch of SegResNet and SwinUNETR on the real data, written to a **separate** folder
(`paper_runs_pilot`). It checks GPU memory, data loading and evaluation before you commit 70+ GPU-hours.
Its 1-epoch results are throw-away: delete the pilot folder afterwards.

In [ ]:
PILOT = [f"output_root={OUT}_pilot", f"data.splits_file={OUT}_pilot/splits_seed42.json",
         "train.epochs=1", "train.val_every=1"]
run("splits seg", "--only splits seg_train:seg_segresnet_s42 seg_eval:seg_segresnet_s42 seg_train:seg_swin_unetr_s42",
    more=PILOT)
import pandas as pd
print(pd.read_csv(f"{OUT}_pilot/seg_segresnet_s42/summary_segmentation_test.csv").round(3).to_string(index=False))
# If all steps show [ok]: memory and data are fine. Remove the throw-away pilot folder:
# !rm -rf "{OUT}_pilot"

## 4. Split + SSL pretraining (~11 h on a T4 for 50 epochs)

In [ ]:
run("splits ssl")

## 5. Segmentation benchmark (all models, same budget)

In [ ]:
run("seg")

## 6. Predicted masks + label-free deep features + radiomics

In [ ]:
run("predict features")

## 7. Multi-task classifiers × seeds + explainability
`explain` writes, per experiment (first seed), to `<run>/explain/`: fusion attention maps, token-occlusion
importance (what each modality contributes) and a 3D Grad-CAM figure per test patient and task.

In [ ]:
run("cls explain")

## 8. K-fold late fusion (XGBoost / logistic regression / RF), tables and figures
Writes `paper_tables/` (CSV, Markdown, LaTeX) and `paper_figures/`: training curves, segmentation boxplots
and qualitative overlays, confusion matrices, ROC curves, k-fold bars, XGBoost TreeSHAP per modality.

In [ ]:
run("kfold aggregate figures")

In [ ]:
from IPython.display import Image, Markdown, display


def show(pattern, limit=None, width=700):
    """Display the PNGs under OUT that match a glob pattern, e.g. show("paper_figures/*_roc.png")."""
    pngs = sorted(glob.glob(f"{OUT}/{pattern}"))
    print(f"{len(pngs)} figure(s) match {pattern}" + (f", showing {limit}" if limit and len(pngs) > limit else ""))
    for png in pngs[:limit]:
        display(Markdown(f"**{os.path.relpath(png, OUT)}**"), Image(png, width=width))


display(Markdown(open(f"{OUT}/paper_tables/summary.md", encoding="utf-8").read()))
show("paper_figures/*.png")

### Explainability figures
`explain` runs on the first seed of each classifier and keeps everything on Drive:
- `paper_figures/explain/<experiment>_occlusion_<task>.png` and `_attention_layer<k>.png`: what each input token
  (image, clinical, radiomics) contributes, averaged over the test patients;
- `<experiment>_s42/explain/gradcam/<patient>_<task>_<layer>.png`: one 3D Grad-CAM (axial / coronal / sagittal
  through the tumour) per test patient, task and pooled feature map. About 1,000 in all, so they stay in the
  run folders; the cell below counts them and shows a few. Change the pattern to look at others.

In [ ]:
show("paper_figures/explain/*.png")
for run_dir in sorted(glob.glob(f"{OUT}/cls_*_s42")):
    n = len(glob.glob(f"{run_dir}/explain/gradcam/*.png"))
    print(f"{os.path.basename(run_dir):35s} {n:4d} Grad-CAM figures" + ("" if n else '   <- none yet: run("explain")'))
show("cls_image_clinical_radiomics_s42/explain/gradcam/*_diagnosis_*.png", limit=4)